### 0 - Modules and Paths

In [1]:
import sys
from pathlib import Path

# add v1.1/ to sys.path for imports to work
sys.path.append(str(Path("..").resolve()))  # if notebook is in notebooks/

# import utils
import modules.yaml_utils as yu
import modules.graph as g
import modules.factory as f

# important paths

ROOT = Path("..").resolve()
FILES = ROOT / "files"
INPUTS = FILES / "inputs"
GENERATED = FILES / "generated"


### 1 - Test Factory - Distances and Paths

In [2]:
graph_yaml = INPUTS / "graph.yaml"
factory_components_yaml = INPUTS / "factory_components.yaml"

graph_dict = yu.load_file(graph_yaml)
print(graph_dict.keys())
factory_components_dict = yu.load_file(factory_components_yaml)

factory = f.FactoryModel(graph_dict, factory_components_dict)

# shortest distances and paths
sd = factory.graph.distance(31, 1)
sp = factory.shortest_path(31, 1, coords = False)
spc = factory.shortest_path_compact(31, 1, coords = False)
sp_coords = factory.shortest_path(31, 1, coords = True)
spc_coords = factory.shortest_path_compact(31, 1, coords = True)

print("31 -> 1:")
print(f"Distance: {sd}m")
print(f"Path: {sp}")
print(f"Compact Path: {spc}")
print(f"Path with coords: {sp_coords}")
print(f"Compact Path with coords: {spc_coords}")


dict_keys(['points_map', 'adj'])
31 -> 1:
Distance: 0.859m
Path: [31, 23, 16, 9, 1]
Compact Path: [31, 16, 9, 1]
Path with coords: [(31, (-0.695, -0.355)), (23, (-0.695, -0.15)), (16, (-0.695, -0.0)), (9, (-0.545, 0.233)), (1, (-0.545, 0.46))]
Compact Path with coords: [(31, (-0.695, -0.355)), (16, (-0.695, -0.0)), (9, (-0.545, 0.233)), (1, (-0.545, 0.46))]


### 1.1 - Test Factory - Initial State, valid actions, update state, terminal state

In [3]:
def print_state_dict(state_dict):
    output_string = "state dict: {"
    for k, v in state_dict.items():
        if k == "boxes":
            for k1, v1 in v.items():
                output_string += f"{k1}: {f.BT[v1]}" + " , "
        elif k == "robot_boxtype":
            output_string += f"{k}: {f.BT[v]}" + " , "
        else:
            output_string += f"{k}: {v}" + " , "
    output_string = output_string[:-3]
    output_string += "}"
    print(output_string)



boxtypes = [f.EMPTY, f.TYPE_A, f.TYPE_B, f.TYPE_C]
initial_state = factory.initial_state(boxtypes)
initial_state_dict = factory.state2dict(initial_state)
initial_state_recovered = factory.dict2state(initial_state_dict)
print(initial_state)
print_state_dict(initial_state_dict)
print(initial_state_recovered)

(31, -1, (-1, 0, 1, 2, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1))
state dict: {robot_node_id: 31 , robot_boxtype: EMPTY , 1: TYPE_A , 2: TYPE_B , 3: TYPE_C}
(31, -1, (-1, 0, 1, 2, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1))


In [4]:
state = initial_state
state_dict = factory.state2dict(state)
print_state_dict(state_dict)
while not factory.terminal_state(state):
    # pick the first valid destination node
    valid_nodes = factory.valid_destinations(state)
    print(f"valid nodes: {valid_nodes}")
    node_to = valid_nodes[0]
    print(f"node to: {node_to}")
    # update state
    state = factory.update_state(state, node_to)
    state_dict = factory.state2dict(state)
    print_state_dict(state_dict)

state dict: {robot_node_id: 31 , robot_boxtype: EMPTY , 1: TYPE_A , 2: TYPE_B , 3: TYPE_C}
valid nodes: [1, 2, 3]
node to: 1
state dict: {robot_node_id: 1 , robot_boxtype: TYPE_A , 1: TYPE_A , 2: TYPE_B , 3: TYPE_C}
valid nodes: [17, 24]
node to: 17
state dict: {robot_node_id: 17 , robot_boxtype: EMPTY , 2: TYPE_B , 3: TYPE_C , 18: TYPE_B}
valid nodes: [2, 3, 18]
node to: 2
state dict: {robot_node_id: 2 , robot_boxtype: TYPE_B , 2: TYPE_B , 3: TYPE_C , 18: TYPE_B}
valid nodes: [13, 20]
node to: 13
state dict: {robot_node_id: 13 , robot_boxtype: EMPTY , 3: TYPE_C , 18: TYPE_B , 14: TYPE_C}
valid nodes: [3, 18, 14]
node to: 3
state dict: {robot_node_id: 3 , robot_boxtype: TYPE_C , 3: TYPE_C , 18: TYPE_B , 14: TYPE_C}
valid nodes: [35, 36, 37, 38]
node to: 35
state dict: {robot_node_id: 35 , robot_boxtype: EMPTY , 18: TYPE_B , 14: TYPE_C , 35: TYPE_C}
valid nodes: [18, 14]
node to: 18
state dict: {robot_node_id: 18 , robot_boxtype: TYPE_B , 18: TYPE_B , 14: TYPE_C , 35: TYPE_C}
valid node